In [1]:
!pip install --upgrade google-meridian[colab,and-cuda,schema]

import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.analysis.review import reviewer
from meridian.data import data_frame_input_data_builder
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
from meridian.schema.serde import meridian_serde
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
from itertools import permutations
from google.colab import drive
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp
import sys
import os

# Mount a storage
drive_mount = '/content/drive'
drive.mount(drive_mount, force_remount=True)
subfolder = '' # @param {"type":"string","placeholder": "Optional, specifying a subfolder is recommended for organizing distinct execution runs."}
# Change this "MyDrive" to other share folders name if you would like to use a different drive.
meridian_root = f'{drive_mount}/MyDrive/{subfolder}'
is_enterprise_user=False

!git clone --branch meridian_modeling https://github.com/pstat197/BlueAlpha3-Synergy-Analysis

df = pd.read_csv("/content/BlueAlpha3-Synergy-Analysis/data/monthly_mocha.csv")
df = df.loc[:, (df != 0).any()]

import sys
sys.path.append("/content/BlueAlpha3-Synergy-Analysis/scripts")
from geometric_mean import create_geometric_mean_interactions

df_mmm = create_geometric_mean_interactions(df)

print(df_mmm.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.3/491.3 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.1/935.1 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.4/718.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 

In [2]:
df_mmm = df_mmm.rename(columns={'date': 'time'}).sort_values(by='time', ascending=True).reset_index(drop=True)

# Convert the 'time' column to datetime objects with the specified format
df_mmm['time'] = pd.to_datetime(df_mmm['time'], format='%m/%d/%y')

# Extract columns with '_x_' and '_spend' for interactions first
interactions = [col for col in df_mmm.columns if '_x_' in col and '_spend' in col]

# define nuisance interaction impressions for filtering
interactions_impressions = [col for col in df_mmm.columns if '_x_' in col and '_impressions' in col]

media_spend_cols = [c for c in df_mmm.columns if c.endswith("_spend") and c not in interactions]
media_impressions_cols = [c for c in df_mmm.columns if c.endswith("_impressions") and c not in interactions_impressions]

media_channels = sorted({
    c.replace("_spend", "").replace("_impressions", "")
    for c in (media_spend_cols + media_impressions_cols)
})

assert len(media_channels) == len(media_spend_cols) == len(media_impressions_cols)

# Create builder
kpi_col = "subscriptions"

builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type="non_revenue",
    default_kpi_column=kpi_col
)

# create revenue_per_unit for revenue_per_kpi
df_mmm['revenue_per_unit'] = 100

builder = (
    builder
    .with_kpi(df_mmm)
    .with_revenue_per_kpi(
        df_mmm,
        revenue_per_kpi_col='revenue_per_unit'
    )
    .with_media(
        df_mmm,
        media_cols=media_impressions_cols,
        media_spend_cols=media_spend_cols,
        media_channels=media_channels
    )
    .with_non_media_treatments(
        df_mmm,
        non_media_treatment_cols=interactions
    )
)


mmm_data = builder.build()

In [3]:
df_mmm

,time,subscriptions,meta_spend,meta_impressions,google_spend,google_impressions,snapchat_spend,snapchat_impressions,tiktok_spend,tiktok_impressions,...,liveintent_impressions,beehiiv_spend,beehiiv_impressions,amazon_spend,amazon_impressions,amazon_spend_x_meta_spend,amazon_impressions_x_meta_impressions,google_spend_x_liveintent_spend,google_impressions_x_liveintent_impressions,revenue_per_unit
0,2025-01-13,10038,0.000000,0,43688.48100,2702666,87326.89737,3163230,39879.67981,1865511,...,92462,0.0,0,0.0,0,0.0,0.0,19836.006276,499893.892433,100
1,2025-01-20,10009,0.000000,0,58085.28804,1787087,80250.59863,3295599,34493.08532,1974043,...,95641,0.0,0,0.0,0,0.0,0.0,23480.611341,413423.254991,100
2,2025-01-27,10146,0.000000,0,78226.85288,4614412,65711.63698,2648478,32828.91921,2010294,...,173350,0.0,0,0.0,0,0.0,0.0,29611.317275,894375.938965,100
3,2025-01-06,8427,0.000000,0,55625.25901,2241951,81736.41300,2638831,32085.57337,1090898,...,82253,0.0,0,0.0,0,0.0,0.0,22477.713984,429426.589306,100
4,2024-10-14,12224,47462.364420,14260449,93423.33719,3176360,46130.90608,2236731,25161.84292,1161739,...,115336,0.0,0,0.0,0,0.0,0.0,28578.972638,605267.425986,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,2024-09-16,10997,0.000000,0,92402.98865,5392627,43282.51664,1656577,20546.65189,1050746,...,104111,0.0,0,0.0,0,0.0,0.0,25579.816012,749287.521314,100
70,2024-09-02,9639,0.000000,0,98821.76918,5811685,37044.97732,1708534,13528.91540,604718,...,119314,0.0,0,0.0,0,0.0,0.0,30251.515109,832715.668215,100
71,2024-09-23,9981,0.000000,0,98195.19441,4761152,33613.49537,1976802,27881.86676,1444044,...,77445,0.0,0,0.0,0,0.0,0.0,28018.265955,607229.294945,100
72,2024-09-30,9725,5760.102132,1028163,102300.40930,5398435,37395.06249,1449038,24583.29996,904567,...,122504,0.0,0,0.0,0,0.0,0.0,29308.073844,813221.913896,100


In [11]:
prior = prior_distribution.PriorDistribution(
    # Media channels:
    roi_m=tfp.distributions.LogNormal(
        loc=0.2,
        scale=0.9
    ),

    # Non-Media (Synergy):
    # Dynamically set location and scale terms based on the number of interactions
    gamma_n=tfp.distributions.Normal(
          loc=[0.0] * len(interactions),
          scale=[0.1] * len(interactions)
    )
)

# Prepare holdout id for testing data
n_geos = 1
n_times = len(df_mmm.time)

np.random.seed(42)
test_pct = 0.2
num_holdout = int(n_times * test_pct)

holdout_id = np.full((n_geos, n_times), False)

holdout_indices = np.random.choice(n_times, size=num_holdout, replace=False)
holdout_id[0, holdout_indices] = True
holdout_id = holdout_id.flatten()

print(f"Created holdout_id with shape: {holdout_id.shape}")
print(f"Number of holdout periods: {np.sum(holdout_id)}")

model_spec = spec.ModelSpec(
    prior=prior,
    enable_aks=True,
    holdout_id=holdout_id,
    media_prior_type='roi',
    non_media_treatments_prior_type='contribution'
)

mmm = model.Meridian(
    input_data=mmm_data,
    model_spec=model_spec
)

# Sample from the model
mmm.sample_prior(500)
mmm.sample_posterior(
    n_chains=10, n_adapt=2000, n_burnin=500, n_keep=1000, seed=0
)

Created holdout_id with shape: (74,)
Number of holdout periods: 14


/usr/local/lib/python3.12/dist-packages/meridian/model/model.py:74: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1325: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. tau_g_excl_baseline has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1325: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_m has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1325: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_rf has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-pa

In [12]:
# Model diagnostics
health_summary = reviewer.ModelReviewer(mmm).run()

filename = 'health_card.html'
health_summary.output_model_health_card(filename=filename, filepath=meridian_root)
IPython.display.HTML(filename=f'{meridian_root}{filename}')

/tmp/ipykernel_15523/1500263722.py:2: DeprecationWarning: The `meridian` argument is deprecated. Please use `model_context` and `inference_data` instead.
  health_summary = reviewer.ModelReviewer(mmm).run()
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:695: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(


Metric check,Status,Recommended action
Convergence,Pass,"The model has likely converged, as all parameters have R-hat values < 1.2."
Baseline,Pass,The posterior probability that the baseline is negative is 0.00. We recommend visually inspecting the baseline time series in the Model Fit charts to confirm this.
Bayesian p-value,Pass,The Bayesian posterior predictive p-value is 0.83. The observed total outcome is consistent with the model's posterior predictive distribution.
Goodness of fit,Pass,"R-squared = 0.8930 (All), 0.9042 (Train), 0.8555 (Test); MAPE = 0.0488 (All), 0.0426 (Train), 0.0758 (Test); wMAPE = 0.0485 (All), 0.0433 (Train), 0.0704 (Test). These goodness-of-fit metrics are intended for guidance and relative comparison."
Prior-posterior shift,Pass 8/8 channels passed,The model has successfully learned from the data. This is a positive sign that your data was informative.
ROI consistency,Pass 8/8 channels passed,"The posterior distribution of the ROI is within a reasonable range, aligning with the custom priors you provided."


In [13]:
# Two-page summary
mmm_summarizer = summarizer.Summarizer(mmm)

filepath = meridian_root
start_date = str(df_mmm["time"].min().date())
end_date = str(df_mmm["time"].max().date())
mmm_summarizer.output_model_results_summary(
    'summary_output.html', filepath, start_date, end_date
)

IPython.display.HTML(filename=f'{meridian_root}/summary_output.html')

/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:695: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:695: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:3356: UserWarning: Effectiveness is not reported because it does not have a clear interpretation by time period.
  warnings.warn(


Dataset,R-squared,MAPE,wMAPE
Training Data,0.90,4%,4%
Testing Data,0.86,8%,7%
All Data,0.89,5%,5%
